In [85]:
# Imports et lecture des données
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

np.random.seed(42)

In [ ]:
# Récupération des fichiers *2021.csv
from pathlib import Path

files_2021 = sorted(Path("../data/cleaned").rglob("*_2021.csv"))

print("Nb de fichiers 2021 trouvés:", len(files_2021))
for p in files_2021:
    print("-", p.as_posix())

assert len(files_2021) > 0, "Aucun CSV 2021 trouvé !"

Nb de fichiers 2021 trouvés: 8
- ../data/cleaned/cleaned2021/cleaned_75_2021.csv
- ../data/cleaned/cleaned2021/cleaned_77_2021.csv
- ../data/cleaned/cleaned2021/cleaned_78_2021.csv
- ../data/cleaned/cleaned2021/cleaned_91_2021.csv
- ../data/cleaned/cleaned2021/cleaned_92_2021.csv
- ../data/cleaned/cleaned2021/cleaned_93_2021.csv
- ../data/cleaned/cleaned2021/cleaned_94_2021.csv
- ../data/cleaned/cleaned2021/cleaned_95_2021.csv


In [87]:
# Chargement des données
df = pd.concat([pd.read_csv(p) for p in files_2021], ignore_index=True)

In [88]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93075 entries, 0 to 93074
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   transaction_date    93075 non-null  object 
 1   property_value      93075 non-null  object 
 2   postal_code         93075 non-null  float64
 3   town_name           93075 non-null  object 
 4   department_code     93075 non-null  int64  
 5   town_code           93075 non-null  int64  
 6   property_type_code  93075 non-null  float64
 7   property_type       93075 non-null  object 
 8   building_area       93075 non-null  float64
 9   main_rooms          93075 non-null  float64
 10  land_area           93075 non-null  float64
dtypes: float64(5), int64(2), object(4)
memory usage: 7.8+ MB


In [ ]:
# Nettoyage de property_value
# La colonne property_value est actuellement de type object (chaîne de caractères)
# car les valeurs contiennent une virgule comme séparateur décimal (ex: "194500,00").
# Cela peut empêcher toute opération mathématique ou modélisation.
# On remplace donc les virgules par rien, puis on convertit en float pour rendre
# les valeurs exploitables par les algorithmes de Machine Learning.
# Harmonisation des types
if df["property_value"].dtype == "object":
    df["property_value"] = df["property_value"].str.replace(" ", "", regex=False).str.replace(",", ".", regex=False).astype(float)

In [93]:
# date -> datetime + year + month
df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors="coerce")
df = df.dropna(subset=["transaction_date"]).copy()

df["year"] = df["transaction_date"].dt.year.astype("Int64")
df["month"] = df["transaction_date"].dt.month.astype("Int64")

In [95]:
# Nettoyage simple des outliers
# limiter l’influence des transactions ultra-chères qui tirent les erreurs vers le haut.
# on coupe au 99,5e percentile (valeur à ajuster selon la distribution).

upper = df["property_value"].quantile(0.995)
df = df[df["property_value"] <= upper].copy()

# Afin de garantir la robustesse du modèle, les valeurs de prix les plus extrêmes
# (au-delà du 99,5e percentile) ont été retirées.
# Ces observations, représentant moins de 0,5 % des données, peuvent biaiser les
# métriques globales (MAE, RMSE) et fausser la calibration du modèle.
# Ce traitement, couramment employé dans la modélisation immobilière, permet de
# concentrer l’apprentissage sur le comportement général du marché tout en limitant
# l’influence des transactions atypiques.

In [96]:
# Variables explicatives & cible
y = df["property_value"].copy()  # prix total en euros

cat_cols = [
    "postal_code",
    "department_code",
    "town_code",
    "property_type_code",
    "town_name",
    "property_type",
]

In [97]:
# Variables numériques structurantes
num_cols = ["building_area", "main_rooms", "land_area", "year", "month"]

X = df[cat_cols + num_cols].copy()

In [98]:
# Train / validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [99]:
# Version log de la cible pour les modèles d’arbres
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

In [100]:
# Préprocesseurs
# Pour les modèles linéaires : standardisation des numériques
preproc_linear = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols),
    ],
    remainder="drop",
)

In [101]:
# Pour les arbres : pas besoin de scaler les numériques
preproc_tree = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop",
)

In [ ]:
#  Modèles

In [102]:
# Modèles linéaires (entraînement directement en euros)
linear_models = {
    "Ridge": Pipeline(
        steps=[
            ("preproc", preproc_linear),
            ("model", Ridge(alpha=1.0)),
        ]
    ),
    "Lasso": Pipeline(
        steps=[
            ("preproc", preproc_linear),
            ("model", Lasso(alpha=0.001, max_iter=5000)),
        ]
    ),
}

In [113]:
# Modèles d’arbres (entraînement en log)
tree_models = {
    "DecisionTree": Pipeline(
        steps=[
            ("preproc", preproc_tree),
            ("model", DecisionTreeRegressor(random_state=42)),
        ]
    ),
    "RandomForest": Pipeline(
        steps=[
            ("preproc", preproc_tree),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=300,
                    min_samples_split=5,
                    min_samples_leaf=1,
                    max_features="sqrt",
                    max_depth=None,
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
    "GradientBoosting": Pipeline(
        steps=[
            ("preproc", preproc_tree),
            ("model", GradientBoostingRegressor(random_state=42)),
        ]
    ),
}

In [109]:
# Fonctions d’évaluation
def evaluate_linear(pipe, X_tr, y_tr_euros, X_va, y_va_euros):
    """
    Modèles linéaires entraînés directement sur les prix en euros.
    Ici pas de log, les métriques restent dans l’échelle originale.
    """
    pipe.fit(X_tr, y_tr_euros)
    y_pred = pipe.predict(X_va)

    mae = mean_absolute_error(y_va_euros, y_pred)
    rmse = np.sqrt(mean_squared_error(y_va_euros, y_pred))
    r2 = r2_score(y_va_euros, y_pred)

    return mae, rmse, r2

In [112]:
def evaluate_tree(pipe, X_tr, y_tr_log, X_va, y_va_euros):
    """
    Modèles d’arbres entraînés sur log(prix).
    Les métriques sont calculées en euros après retour à l’échelle initiale.
    """
    pipe.fit(X_tr, y_tr_log)
    y_pred_log = pipe.predict(X_va)
    y_pred = np.expm1(y_pred_log)

    mae = mean_absolute_error(y_va_euros, y_pred)
    rmse = np.sqrt(mean_squared_error(y_va_euros, y_pred))
    r2 = r2_score(y_va_euros, y_pred)

    return mae, rmse, r2

In [ ]:
#  Les modèles linéaires sont sensibles à la distribution de la cible.
#   Quand on entraîne un modèle linéaire directement sur les prix en euros,
#   il suppose une relation linéaire entre X et y et des résidus à peu près symétriques.
#   Conserver l’échelle originale permet de garder une interprétation simple des coefficients
#   et d’obtenir des métriques directement exprimées en euros.
#
#  À l’inverse, les modèles d’arbres (Random Forest, Gradient Boosting, XGBoost, etc.)
#   capturent très bien les non-linéarités, mais sont plus sensibles aux valeurs extrêmes.
#   Or les prix des matériaux critiques sont souvent très asymétriques (skewed)
#   et comportent de grandes variations ou des "spikes".
#
# Appliquer log(y) pour les modèles d’arbres permet :de réduire l’impact des valeurs
# extrêmes, de stabiliser la variance,de rendre la distribution de y plus proche
# d’une loi normale,d’améliorer la capacité du modèle à apprendre des variations
# relatives plutôt qu’absolues.

In [110]:
# Résultats : modèles linéaires

rows_lin = []
for name, pipe in linear_models.items():
    mae, rmse, r2 = evaluate_linear(pipe, X_train, y_train, X_val, y_val)
    rows_lin.append({"Model": name, "MAE (€)": mae, "RMSE (€)": rmse, "R2": r2})

results_linear = pd.DataFrame(rows_lin).sort_values(by="R2", ascending=False).reset_index(drop=True)
results_linear

/workspaces/gpd-m2sep-france-property-insight/.venv/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:656: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.284e+16, tolerance: 3.536e+13
  model = cd_fast.sparse_enet_coordinate_descent(


,Model,MAE (€),RMSE (€),R2
0,Ridge,595299.562504,1.588306e+06,0.466523
1,Lasso,595643.004886,1.588350e+06,0.466493


In [115]:
# Résultats : modèles d’arbres


rows_tree = []
for name, pipe in tree_models.items():
    mae, rmse, r2 = evaluate_tree(pipe, X_train, y_train_log, X_val, y_val)
    rows_tree.append({"Model": name, "MAE (€)": mae, "RMSE (€)": rmse, "R2": r2})

results_tree = pd.DataFrame(rows_tree).sort_values(by="R2", ascending=False).reset_index(drop=True)
results_tree

,Model,MAE (€),RMSE (€),R2
0,DecisionTree,251850.724368,9.943436e+05,0.790916
1,RandomForest,277284.607884,1.039178e+06,0.771636
2,GradientBoosting,481277.604901,1.705304e+06,0.385034
